<a href="https://colab.research.google.com/github/starlton/Deep-Learning/blob/main/Week%202/week2_micrograd_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 — Building micrograd: A Tiny Autograd Engine

**Goal:** Build an automatic differentiation engine from scratch — the same core idea that powers PyTorch and TensorFlow.

Autograd (automatic differentiation) is what lets a neural network compute the gradient of its loss with respect to every parameter *automatically*, no matter how complex the network. We'll build a minimal version that handles scalar values.

**The big idea:** Every math expression can be represented as a **computational graph**. Each node holds a value. To train, we need the gradient of the output with respect to every input — and we get it by walking the graph *backward*, applying the **chain rule** at each node.

**What we'll build:**
1. A `Value` class that wraps a number and tracks how it was computed
2. `__add__` and `__mul__` operations that build the graph and store local gradient rules
3. A `backward()` method that propagates gradients through the whole graph

---

## Setup — Imports

In [1]:
import numpy as np
import math
import matplotlib.pyplot as plt

%matplotlib inline

---
## The Theory — Computational Graphs and the Chain Rule

Consider the expression we'll use as our running example:

$$f(a, b, c) = (a + b) \cdot c$$

We break it into intermediate steps, each a single operation:

$$u = a + b \qquad v = c \cdot u \qquad (v = f)$$

This forms a **computational graph**:

```
a ──┐
    ├──> (+) ──> u ──┐
b ──┘               ├──> (*) ──> v
c ──────────────────┘
```

### Forward pass (with a=2, b=3, c=4)
$$u = 2 + 3 = 5 \qquad v = 4 \cdot 5 = 20$$

### Backward pass — the chain rule
We want $\frac{\partial v}{\partial a}$, $\frac{\partial v}{\partial b}$, $\frac{\partial v}{\partial c}$.

Each operation has a **local gradient** — how its output changes with respect to each input:

**Addition node** ($u = a + b$): the gradient passes through *unchanged*.
$$\frac{\partial u}{\partial a} = 1 \qquad \frac{\partial u}{\partial b} = 1$$

**Multiplication node** ($v = c \cdot u$): each input's gradient is the *other* input's value.
$$\frac{\partial v}{\partial c} = u = 5 \qquad \frac{\partial v}{\partial u} = c = 4$$

**Chaining back to the leaves** using $\frac{\partial v}{\partial a} = \frac{\partial v}{\partial u} \cdot \frac{\partial u}{\partial a}$:

$$\frac{\partial v}{\partial a} = 4 \cdot 1 = 4 \qquad \frac{\partial v}{\partial b} = 4 \cdot 1 = 4 \qquad \frac{\partial v}{\partial c} = 5$$

These are the three numbers our code must reproduce: **a.grad=4, b.grad=4, c.grad=5**.

---

## The `Value` Class

Each `Value` wraps a single number and remembers:

| Attribute | Purpose |
|---|---|
| `data` | The actual number |
| `grad` | The gradient of the final output w.r.t. this value (starts at 0) |
| `_children` | The Values that produced this one (the graph edges) |
| `_backward` | A function that pushes this node's gradient to its children |

### How each operation stores its gradient rule

When we compute `out = self + other`, we attach a `_backward` closure to `out`. That closure knows the **local gradient rule** for addition and applies the chain rule:

**Addition** — gradient flows through unchanged:
```
self.grad  += out.grad      # local gradient is 1
other.grad += out.grad      # local gradient is 1
```

**Multiplication** — gradient scaled by the *other* operand:
```
self.grad  += other.data * out.grad
other.grad += self.data  * out.grad
```

Note the `+=` (not `=`): a value used in multiple places accumulates gradient from every path. This is the multivariable chain rule.

### The `backward()` method

1. **Topological sort** — order nodes so each comes after its inputs
2. **Seed** the output: $\frac{\partial v}{\partial v} = 1$
3. **Walk in reverse**, calling each node's `_backward()` so gradient flows from output to inputs

In [2]:
class Value:
  def __init__(self, data, _children=()): #grad, children, operation):
    self.data = data
    self.grad = 0.0 ## Condition
    self._children = set(_children)
    self._backward = lambda: None

  def __repr__(self):
    return f"Value=({self.data})"


  def __add__(self, other):
    out = Value(self.data + other.data, _children=(self, other))
    def _backward():
      self.grad +=  out.grad
      other.grad += out.grad

    out._backward = _backward
    return out

  def __mul__(self, other):
    out = Value(self.data * other.data, _children=(self, other))

    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad

    out._backward = _backward
    return out

  def backward(self):
    # 1. Topological order all of the children in the graph
    topo = []
    visited = set()
    def build_topo(node):
      if node not in visited:
        visited.add(node)
        for child in node._children:
          build_topo(child)

        topo.append(node)

    build_topo(self)

    # 2. Seed the output gradient
    self.grad = 1.0

    # 3. Apply the chain rule in reverse topological order
    for node in reversed(topo):
      node._backward()


### Walking through `backward()`

**Why topological sort?** Gradient can only flow *into* a node once all the nodes *downstream* of it (closer to the output) have computed their gradients. Topological order guarantees that when we reverse it, we always process a node before the nodes that feed into it.

**Why seed with 1.0?** The gradient of the output with respect to itself is trivially 1: $\frac{\partial v}{\partial v} = 1$. This is the spark that starts the chain.

**Why reversed order?** We start at the output (gradient = 1) and push gradient backward toward the inputs, one operation at a time — exactly the backward pass we did by hand.

---

## Testing — Recreate the Paper Example in Code

$$f(a,b,c) = (a + b) \cdot c \quad\text{with}\quad a=2,\; b=3,\; c=4$$

Expected results from our hand derivation:
- `f.data` = 20 (forward pass)
- `a.grad` = 4, `b.grad` = 4, `c.grad` = 5 (backward pass)
- `f.grad` = 1 (the seed)

In [3]:
a = Value(2.0)
b = Value(3.0)
c = Value(4.0)

f = (a+b) * c


f.backward()

a.grad, b.grad, c.grad, f.grad

(4.0, 4.0, 5.0, 1.0)

### Result: `(4.0, 4.0, 5.0, 1.0)` ✓

The code reproduces our hand-derived gradients exactly:

| Gradient | By hand | micrograd | Match |
|---|---|---|---|
| $\partial f / \partial a$ | 4 | 4.0 | ✓ |
| $\partial f / \partial b$ | 4 | 4.0 | ✓ |
| $\partial f / \partial c$ | 5 | 5.0 | ✓ |
| $\partial f / \partial f$ | 1 | 1.0 | ✓ |

**We have a working automatic differentiation engine.** It takes any expression built from `+` and `*`, builds the computational graph automatically, and computes the gradient of the output with respect to every input in a single backward pass.

---

## What's Next

This is the seed of every deep learning framework. To grow it into something that can train a neural network, the next steps (coming days) are:

- Add more operations: `__pow__`, `__sub__`, `tanh`, `relu`, `exp`
- Handle operations with constants (e.g. `Value(2) + 1`)
- Build `Neuron`, `Layer`, and `MLP` classes on top of `Value`
- Train the MLP on a real problem (XOR) using gradient descent